In [0]:
%fs ls dbfs:/databricks-datasets/amazon/


path,name,size,modificationTime
dbfs:/databricks-datasets/amazon/README.md,README.md,396,1456789691000
dbfs:/databricks-datasets/amazon/data20K/,data20K/,0,0
dbfs:/databricks-datasets/amazon/test4K/,test4K/,0,0
dbfs:/databricks-datasets/amazon/users/,users/,0,0


In [0]:
df=spark.read.parquet('dbfs:/databricks-datasets/amazon/data20K/')

In [0]:
import pyspark.sql.types as T
import pyspark.sql.functions as F
import pyspark.sql.window as win

import data

In [0]:
df.schema

Out[3]: StructType([StructField('rating', DoubleType(), True), StructField('review', StringType(), True)])

In [0]:
df.columns

Out[4]: ['rating', 'review']

In [0]:
df1=df.selectExpr('rating','explode(split(trim(review), " ")) as words')

In [0]:
df1=df.select('rating', F.explode(F.split(F.trim(F.col('review'))," ")).alias("words"))

In [0]:
display(df1.head(10))

rating,words
4.0,Worked
4.0,as
4.0,expected.I'm
4.0,not
4.0,sure
4.0,what
4.0,else
4.0,you
4.0,expect
4.0,me


In [0]:
df2=df1.select("rating", F.explode(F.split(F.trim(F.col('words')), '\.')).alias("words")).select('rating', 'words', F.length('words').alias("word_len")).filter('word_len >0')

In [0]:
display(df2)

rating,words,word_len
4.0,Worked,6
4.0,as,2
4.0,expected,8
4.0,I'm,3
4.0,not,3
4.0,sure,4
4.0,what,4
4.0,else,4
4.0,you,3
4.0,expect,6


In [0]:
df3=df2.groupBy('words').agg(F.count(F.col('rating')).alias("word_count"),F.avg(F.col('rating')).alias("avg_rating"))

In [0]:
display(df1.head(10))

rating,words
4.0,Worked
4.0,as
4.0,expected.I'm
4.0,not
4.0,sure
4.0,what
4.0,else
4.0,you
4.0,expect
4.0,me


In [0]:
display(df3.orderBy([F.col("avg_rating")], ascending=[False]))

words,word_count,avg_rating
"tooo,",1,5.0
joggers,1,5.0
stab,2,5.0
Comply,1,5.0
clumps!!,1,5.0
IT!!!,3,5.0
dishwater,1,5.0
yourselves!,1,5.0
bendable,1,5.0
"genuine,",1,5.0


In [0]:
display(df1.head(10))

rating,words
4.0,Worked
4.0,as
4.0,expected.I'm
4.0,not
4.0,sure
4.0,what
4.0,else
4.0,you
4.0,expect
4.0,me


In [0]:
display(df.head(10))

rating,review
4.0,Worked as expected.I'm not sure what else you expect me to say. I expected no less.Dunno what else to say.
5.0,"This mouse is amazing, I had owned a Razer Naga, and a R.A.T. 7 mouse. And I am not afraid to say that this mouse takes the cake, for $20 it looks like its worth $80. The only issue i had was that the seller sent me a wireless version, but I don't mind because it works just as well. Also where you see silver on the mouse in reality it is glossy black, and the mouse looks better IRL then in the pictures. I will add on to this review in a month or so to check in."
4.0,we recently had a baby boy so now the use of my home theater system is on the shelf for awhile. My wife also is not into big sound so decided to try out these wireless headphones. at first i almost sent them back because i took the easy route and hooked them into my receivers headphone jack. The sound was terrible and there was the dreaded hissing noise you get with some wireless electronics. before packing them up i decided to hook them up into the jacks on the TV. Difference was like night and day. the sound quality is great and really brings out some of the background sounds and music that you typically don't notice unless you are in a movie theater. They aren't super comfortable for long durations but that can be expected with most headphones. these come fully recommended.
3.0,"Works good for a boy of 6 and his parents.This game is not really requiring any skills (be it game skills, memory or else). It is still fun but gets repetitive fast. Said that, my son still enjoyed it enough to want a proper ""adult"" Catan."
2.0,"Fabric is nice and soft but zipper broke the first time we used it. Very disappointing. The fit was fine, so we still use it to get our monies worth"
4.0,"They are screws with square heads, what else do you need to know. They work, they hold two pieces of wood together."
5.0,I love this prolduct and the other items that go with it. Bath and Body dropped this scent so I as thrilled find that I could still purchase it.
3.0,"This is an ok pump. It has the nice bracket that can mount to your bike and you have it for the road.And although the image here doesn't show it, it does have the gauge, though it isn't the most accurate gauge (not really a problem when the tire is rated for 55 to 100 PSI).Much like one other reviewer noted, the instructions on how to switch from Schrader to Presta are not very clear and should be researched before fumbling with this blasted thing for hours trying to make it work.And like the title mentions, this isn't as tiny as the MINI name implies. When looking for a place to mount it on my bike, I am really running short on ideas. The obvious places would interfere with pedaling.All that said, it puts air in the tire. Slowly. It is double action, but the throw isn't very much. And the higher the pressure, the more difficult it is to pump."
5.0,"Perfect for the gym or under a few layers for the chilly morning runs. I own quite a few, and just can't get enough!"
4.0,"I'd hoped my 2 year old kitty would have been more enthusiastic about these little Chew Mice, but perhaps she has too many other toys. All-in-all, they're a good addition to her toy collection."
